In [ ]:
# Script for styling the Jupyter Notebook using an external CSS file
from IPython.display import HTML

with open("style.css", "r") as f:
    css = f.read()

HTML(f"<style>{css}</style>")

# Control analysis


In [ ]:
# Control examples
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

# Generate functions
def boosting(n_specs= 4, k = None):
    p_noint   = np.random.uniform(0, 0.9)
    # Remove diagonal
    total     = n_specs**2 - n_specs
    num_noint = int(np.floor(p_noint * total))
    num_negs  = int(total - num_noint)
    # Interaction matrix
    species_names = [f"S{i}" for i in range(1, n_specs + 1)]
    M = np.full((n_specs, n_specs), np.nan)
    np.fill_diagonal(M, -0.5)
    # Build and shuffle off-diagonal values
    interaction_values = np.random.permutation(np.concatenate([
        np.zeros(num_noint),
        -np.random.uniform(0, 1, num_negs)
    ]))
    # Fill off-diagonal elements (row by row, skipping diagonal)
    off_diag_mask = ~np.eye(n_specs, dtype=bool)
    M[off_diag_mask] = interaction_values
    # Scale column k (converting 1-based R index to 0-based Python index)
    M[:, k] *= 10
    M = np.round(M, 5)
    # Wrap in a labeled DataFrame
    M_df = pd.DataFrame(M, index=species_names, columns=species_names)
    edges_to_highlight = pd.DataFrame({
        "from": f"S{k}",
        "to":   [f"S{i}" for i in range(1, n_specs + 1) if i != k ]
    })
    return M_df, edges_to_highlight

# Define cascade method
def cascade(n_specs= 4, k = None):
    p_noint   = np.random.uniform(0, 0.9)
    # Remove diagonal and n - 1 interactions
    total     = (n_specs - 1) ** 2
    num_noint = int(np.floor(p_noint * total))
    num_negs  = int(total - num_noint)
    # interaction values — permutation replaces concatenate + shuffle
    interaction_values = np.random.permutation(np.concatenate([
        np.zeros(num_noint), -np.random.uniform(0, 1, num_negs)
    ]))
    # cascade sequence — work 0-based directly, drop cascade_idx and c_rows/c_cols
    whom_rows = np.delete(np.arange(n_specs), k)      # [1, 2, 3, 4]
    who_cols  = np.concatenate([[k], whom_rows[:-1]])  # [0, 1, 2, 3]
    # Build M
    node_names = [f"S{i}" for i in range(1, n_specs + 1)]
    M = np.full((n_specs, n_specs), np.nan)
    np.fill_diagonal(M, -0.5)
    M[whom_rows, who_cols] = -np.random.uniform(0, 1, n_specs - 1) * 30
    # Fill not cascade interactions
    mask = ~np.eye(n_specs, dtype=bool)
    mask[whom_rows, who_cols] = False
    M[mask] = interaction_values
    M    = np.round(M, 5)
    M_df = pd.DataFrame(M, index=node_names, columns=node_names)
    # edges_to_highlight: who_cols (from) -> whom_rows (to)
    edges_to_highlight = pd.DataFrame({
        "from": [f"S{s+1}" for s in who_cols],
        "to":   [f"S{t+1}" for t in whom_rows]
    })
    return M_df, edges_to_highlight

# Define magnitude function
def magnitude(n_specs= 4, k = None):
    p_noint   = np.random.uniform(0, 0.9)
    # Interaction counts
    total     = (n_specs - 1) ** 2
    num_noint = int(p_noint * total)
    num_negs  =  total - num_noint 
    # Build matrix
    M = np.full((n_specs, n_specs), np.nan)
    # Precompute mask: off-diagonal AND not in column k 
    off_diag_off_k = ~np.eye(n_specs, dtype=bool) & (np.arange(n_specs) != k)
    M = np.full((n_specs, n_specs), np.nan)
    M[off_diag_off_k] = np.random.permutation(np.concatenate([
        np.zeros(num_noint),
        -np.random.uniform(1, 2, num_negs)
    ]))
    M[:, k] = -np.random.uniform(0, 1, n_specs)
    np.fill_diagonal(M, -0.5)
    # Add names to df
    node_names = [f"S{i}" for i in range(1, n_specs + 1)]
    M_df = pd.DataFrame(M, index=node_names, columns=node_names)
    edges_to_highlight = pd.DataFrame({
        "from": f"S{k + 1}",
        "to":   [f"S{i}" for i in range(1, n_specs + 1) if i != k + 1]
    })
    return M_df, edges_to_highlight

# Create figure
n_specs = 5
k =  np.random.randint(0, n_specs)


# Build graph from M_df (your interaction matrix)
plt.clf()
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, ax in enumerate(axes):
    if i == 0:
        # Boosting method
        M_df, edges_to_highlight = boosting(n_specs, k)
        ax.set_title('Ejemplo de red método de incremento de columna')
        legend_handles = [
            mlines.Line2D([], [], color="gray",    linewidth=1.5, label=r"$\mathcal{U}[-1, 0]$"),
            mlines.Line2D([], [], color="crimson", linewidth=3.5, label=r"$\mathcal{U}[-1, 0] \times 10$"),
        ]
    if i == 1:
        # Cascade method
        M_df, edges_to_highlight = cascade(n_specs, k)
        print(edges_to_highlight)
        ax.set_title('Ejemplo de red método de cascada')
        legend_handles = [
            mlines.Line2D([], [], color="gray",    linewidth=1.5, label=r"$\mathcal{U}[-1, 0]$"),
            mlines.Line2D([], [], color="crimson", linewidth=3.5, label=r"$\mathcal{U}[-1, 0] \times 10$"),
        ]
    if i == 2: 
        # Magnitude method
        M_df, edges_to_highlight = magnitude(n_specs, k)
        ax.set_title('Ejemplo de red método de diferencia de magnitud')
        legend_handles = [
            mlines.Line2D([], [], color="gray",    linewidth=1.5, label=r"$\mathcal{U}[-2, -1]$"),
            mlines.Line2D([], [], color="crimson", linewidth=3.5, label=r"$\mathcal{U}[-1, 0]$"),
        ]
    # Nodes to color
    # Build highlight edge list from your dataframe
    highlight_edges = list(zip(edges_to_highlight["from"], edges_to_highlight["to"]))
    # Generate graph
    G = nx.from_pandas_adjacency(M_df, create_using=nx.DiGraph())
    pos = nx.circular_layout(G)
    highlight_node = f"S{k}"
    node_colors = ["orange" if n == highlight_node else "lightblue" for n in G.nodes()]
    # edges to color
    highlight_edges = [(u, v) for u, v in highlight_edges if u != v and G.has_edge(u, v)]
    normal_edges    = [e for e in G.edges() if e not in highlight_edges]
    # Draw each graph on its own axis
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=900)
    nx.draw_networkx_labels(G, pos, ax=ax)
    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=normal_edges, edge_color="gray", width=1.5, arrows=True)
    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=highlight_edges, edge_color="crimson", width=3.5, arrows=True)
    ax.axis("off")
    # Add legends
    ax.legend(
        handles=legend_handles,
        loc="upper right",
        frameon=True,
        fontsize=10,
    )


plt.savefig(f'/home/mriveraceron/glv-research/thesis_plots/testing_graph.png', dpi = 300)